# Архивный статус 40.90

Полная историческая копия объединённого прототипа сохранена только для
происхождения изменений. Статус — `legacy_unvalidated`; файл содержит прежние
производные значения по добровольцу и не является активным источником данных,
алгоритма или выводов. Вычислительные outputs очищены.


> **Статус миграции:** это пока действующий объединённый источник, а не окончательно разделённый ноутбук.
> 
> Предусмотренное разделение: `40.11`, `40.12`, `40.20` и `40.21`. Полная копия происхождения: `archive/legacy/40.90_Объединённый_прототип_ТТРКГ.ipynb`. Новые каркасы не считаются реализованными до переноса и сравнения вычислений.


# Сердечный сигнал трансторакального отведения и границы лёгочного конфаундера

Ноутбук готовит вход для обратной задачи по геометрии сердца: выделяет сердечно-синхронный сигнал трансторакального отведения и устанавливает, насколько он может быть загрязнён пульсовым кровенаполнением лёгкого.

**Разделение источников данных.** Удельные сопротивления тканей и их пульсовые приращения берутся из эксперимента МГТУ `EXP_2026_04_29` (канал 2, лёгочная ветка [09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb), [11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb)), поскольку это свойства тканей испытуемого, а не прибора. Трансторакальный сигнал берётся из эксперимента РНЦХ `EXP_2026_07_16` (канал 1), поскольку канал 1 прибора МГТУ неисправен по абсолютной величине ([17](10.01_QC_эксперимента_2_порядок_и_каналы.ipynb) §4.2), а в РНЦХ тот же канал исправен. Испытуемый в обеих сессиях один и тот же.

**Наследуемая методика.** Детекция зубцов R и окна электрической систолы — по [08](11.12_Разметка_ЭКГ_эксперимента_3.ipynb) §0. Ансамблевое усреднение и определение систолического размаха (11.1) — по [11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb) §1. Рабочая точка и требование согласовывать состояние дыхания с состоянием ансамбля — по [11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb) §3.4. Свойства сферической модели и ограничения замены сердца сферой — по [18](70.09_Импедансный_критерий_эквивалентной_сферы.ipynb).


## §0. Методика и словарь терминов

### Решаемая задача
Требуется получить сердечно-синхронное изменение импеданса трансторакального отведения в форме, пригодной для обратной задачи по геометрии сердца, и оценить сверху вклад пульсового кровенаполнения лёгкого в этот сигнал. Без такой оценки нельзя判 судить, требуется ли вычитание конфаундера или им можно пренебречь.

### Метод и обоснование выбора
Сердечно-синхронная составляющая выделяется ансамблевым усреднением по зубцам R, как в [11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb). Влияние подключённого соседнего канала измеряется **внутри одной записи**, в которой канал поочерёдно отключался при неизменном положении электродов: это отделяет свойство прибора от свойства установки. Граница лёгочного вклада устанавливается аналитически, из однородности прямой задачи, и потому не требует расчёта методом конечных элементов.

### Словарь терминов
- **Трансторакальное отведение (ТТРКГ)** — канал 1, токовые электроды на руках, ток идёт через грудной объём.
- **Систолический размах** $\Delta Z_{\text{сист}}$ — разность наибольшего и наименьшего значений усреднённой по зубцам R формы пульсового импеданса, Ом; определение (11.1).
- **Относительный размах** $\Delta Z/Z$ — систолический размах, отнесённый к базовому импедансу того же отведения; величина, не зависящая от общего множителя измерительного тракта ([09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb) §3.2).
- **Относительная чувствительность к ткани** $S_i=\dfrac{\rho_i}{Z}\dfrac{\partial Z}{\partial\rho_i}$ — безразмерная доля, с которой удельное сопротивление ткани $i$ входит в измеряемый импеданс.
- **Межканальное влияние** — изменение измеряемой величины одного канала при подключении или отключении другого при неизменных электродах ([17](10.01_QC_эксперимента_2_порядок_и_каналы.ipynb) §4.1).

### Критерии интерпретации
Конфигурация измерения считается соответствующей модели, если в ней работает одна токовая пара: аналитическая модель ([01](70.02_Прямая_модель_эквивалентной_сферы_РЕО32.md)) и расчёт методом конечных элементов второго генератора не описывают. Вычитание конфаундера признаётся необходимым, если его верхняя граница сопоставима с измеренным сигналом или превышает его.


## §1. Извлечение трансторакального сигнала из записей РНЦХ

**Входные данные.** Записи эксперимента 3 из внешнего каталога, заданного в `KALMYKOV_EXP03_CONFIG`; частота дискретизации 200 Гц. Используются столбцы `TIME_s`, `ECG_V`, `BASE_1_Ω`, `RHEO_1_mΩ`, `QS_1_Ω`, а также `BASE_2_Ω` для определения состояния соседнего канала.

**Допущения.** Канал считается включённым, если его базовый импеданс превышает 5 Ом; это правило применялось и в [16](10.11_QC_эксперимента_3.ipynb) §4. Детекция зубцов R выполняется полосовой фильтрацией 5–25 Гц с последующим поиском пиков, как в [08](11.12_Разметка_ЭКГ_эксперимента_3.ipynb); порог подбирается по записи, поскольку качество электрокардиограммы между записями различается. Качество детекции контролируется сопоставлением числа найденных зубцов с длительностью записи: если средняя частота по числу зубцов расходится с медианной частотой по интервалам более чем на четверть, детекция признаётся ненадёжной и ансамбль по такой записи не строится.

Окно ансамбля и отбраковка совпадают с [11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb) §1: от −0.15 до +0.70 с относительно зубца R.


In [ ]:
# @title §1 Загрузка записей РНЦХ, детекция R с контролем качества, ансамбль ТТРКГ
import json, os, glob
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
%matplotlib inline

CONFIG_PATH = Path(os.environ["KALMYKOV_EXP03_CONFIG"]).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
RNCH = Path(CONFIG["data_root"]).expanduser().resolve() / CONFIG.get("csv_subdir", "")
PRE, POST = 0.15, 0.70          # окно ансамбля, с — как в 11 §1
ON_LEVEL  = 5.0                 # порог «канал включён», Ом — как в 16 §4
QS_CEIL   = 4700.0              # потолок шкалы QS, Ом

def detect_r(ecg, fs):
    """Зубцы R по методике 08 §0 с подбором порога и контролем качества."""
    b, a = butter(2, [5/(fs/2), 25/(fs/2)], btype="band")
    f = filtfilt(b, a, ecg); f /= (np.std(f) + 1e-9)
    best = None
    for h in (2.0, 1.5, 1.2, 1.0, 0.8):
        pk, _ = find_peaks(f, distance=int(0.33*fs), height=h, prominence=0.6)
        if len(pk) < 5: continue
        rr = np.median(np.diff(pk/fs))
        hr_int, hr_cnt = 60/rr, len(pk)/(len(ecg)/fs)*60
        # два независимых условия: физиологичность частоты и согласие двух её оценок
        ok = (40 <= hr_int <= 120) and (40 <= hr_cnt <= 120) and abs(hr_int - hr_cnt)/hr_int < 0.25
        if best is None or (ok and not best[3]): best = (pk/fs, hr_int, hr_cnt, ok, h)
        if ok: break
    return best if best else (np.array([]), np.nan, np.nan, False, np.nan)

def ensemble(x, R, fs, t0=None, t1=None):
    if t0 is not None: R = R[(R >= t0) & (R <= t1)]
    ip, ipo = int(PRE*fs), int(POST*fs)
    segs = [x[int(r*fs)-ip:int(r*fs)+ipo] for r in R
            if int(r*fs)-ip >= 0 and int(r*fs)+ipo < len(x)]
    if len(segs) < 3: return None, 0, np.nan
    avg = np.array(segs).mean(0)
    return avg, len(segs), (np.argmin(avg)-ip)/fs

REC = {}
print("%-10s | %6s | %-13s | %8s | %9s | %7s | %s" %
      ("запись", "длит.", "каналы", "BASE_1", "QS_1", "N уд.", "детекция R"))
print("-"*92)
for path in sorted(glob.glob(os.path.join(RNCH, "*.csv"))):
    key = os.path.basename(path)[11:19]
    df = pd.read_csv(path, encoding="utf-8")
    t = df["TIME_s"].values; fs = 1.0/np.median(np.diff(t))
    b1, b2 = df["BASE_1_Ω"].values, df["BASE_2_Ω"].values
    on1, on2 = np.median(b1) > ON_LEVEL, np.median(b2) > ON_LEVEL
    R, hr_i, hr_c, ok, thr = detect_r(df["ECG_V"].values, fs)
    REC[key] = dict(df=df, fs=fs, R=R, ok=ok, on1=on1, on2=on2, t=t)
    print("%-10s | %5.1fс | %-13s | %7.2f | %8.0f | %7d | %s" %
          (key, t[-1], ("оба" if on1 and on2 else ("только 1" if on1 else "только 2")),
           np.median(b1), np.median(df["QS_1_Ω"].values), len(R),
           "надёжна (порог %.1f, ЧСС %.0f)" % (thr, hr_i) if ok else "НЕНАДЁЖНА — ансамбль не строится"))

print()
print("Ансамбль трансторакального канала по записям с надёжной детекцией:")
print("%-10s | %-13s | %6s | %9s | %11s | %9s | %s" %
      ("запись", "каналы", "N уд.", "BASE_1", "ΔZ_сист", "ΔZ/Z", "провал"))
print("-"*92)
for key, r in REC.items():
    if not (r["on1"] and r["ok"]): continue
    avg, n, tr = ensemble(r["df"]["RHEO_1_mΩ"].values, r["R"], r["fs"])
    if avg is None: continue
    Zb = float(np.median(r["df"]["BASE_1_Ω"].values)); dZ = (avg.max()-avg.min())/1000.0
    r.update(Zb=Zb, dZ=dZ, avg=avg, trough=tr)
    print("%-10s | %-13s | %6d | %8.2f Ом | %8.1f мОм | %8.3f %% | %+6.0f мс" %
          (key, "оба" if r["on2"] else "только 1", n, Zb, dZ*1000, 100*dZ/Zb, tr*1000))


**Анализ результатов.** Из четырёх записей трансторакальный канал включён в трёх; запись `14-09-42` из дальнейшего исключается, так как в ней работает только канал 2.

Качество электрокардиограммы между записями различается, и единый порог детекции непригоден: для `14-17-16` он составляет 0.8, для остальных 1.5. Подобранные пороги дают частоту сердечных сокращений 68–82 в минуту, согласованную по двум независимым оценкам — по медианному межударному интервалу и по числу зубцов на длительность записи. Обе оценки нужны совместно: проверка одной лишь их согласованности пропускает случай, когда детектор устойчиво находит лишь малую долю зубцов, поскольку тогда занижены обе.

Ансамбли построены по трём записям. Провал усреднённой волны наступает через 250–355 мс после зубца R, что согласуется по порядку с задержкой, измеренной в лёгочной ветке (320–340 мс, [11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb) §4), и подтверждает сердечное происхождение выделенной составляющей.

Существенно различие между записями по относительному размаху: 0.169 % в записи с одиночной токовой парой против 0.294 и 0.334 % в записях с обоими включёнными каналами. Различие вдвое превышает разброс между однотипными записями и разбирается в §2.

**Результаты и умозаключения.** Пригодны все три записи с включённым трансторакальным каналом. Запись `14-07-01` даёт наилучшую статистику для конфигурации с одиночной токовой парой — 71 удар; запись `14-19-52` содержит полный дыхательный протокол и 69 ударов при обоих включённых каналах; запись `14-17-16` используется в §2 для интервального сопоставления, поскольку только в ней состояние каналов меняется при неизменных электродах.


## §2. Межканальное влияние на базовый и на пульсовой импеданс

**Входные данные.** Запись `14-17-16`, в которой каналы поочерёдно отключались извлечением кабеля из прибора при неизменном положении электродов на испытуемом.

**Допущения.** Интервалы состояний определяются по условию «базовый импеданс канала выше 5 Ом», как в [16](10.11_QC_эксперимента_3.ipynb) §4. Интервалы короче 4 с не рассматриваются. Поскольку электроды не переклеивались, различие между интервалами относится к прибору, а не к установке электродов, и потому является измерением межканального влияния в чистом виде.

Сопоставляются две величины: базовый импеданс и систолический размах. Их отношение показывает, действует ли подключённый соседний канал как простой параллельный шунт, при котором обе величины менялись бы согласованно, или он перераспределяет ток и меняет чувствительность отведения к сердцу.


In [ ]:
# @title §2 Влияние подключённого соседнего канала внутри одной записи
r = REC["14-17-16"]
df, fs, R = r["df"], r["fs"], r["R"]
t = r["t"]; b1, b2 = df["BASE_1_Ω"].values, df["BASE_2_Ω"].values
state = np.where((b1 > ON_LEVEL) & (b2 > ON_LEVEL), 2,
                 np.where((b1 > ON_LEVEL) & (b2 <= ON_LEVEL), 1, 0))
edges = [0] + list(np.where(np.diff(state) != 0)[0] + 1) + [len(state)]

print("%-14s | %-14s | %6s | %10s | %11s | %10s | %s" %
      ("интервал, с", "состояние", "N уд.", "BASE_1", "ΔZ_сист", "ΔZ/Z", "провал"))
print("-"*94)
ROWS = []
for k in range(len(edges)-1):
    i0, i1 = edges[k], edges[k+1]
    if state[i0] == 0 or (t[i1-1] - t[i0]) < 4: continue
    avg, n, tr = ensemble(df["RHEO_1_mΩ"].values, R, fs, t[i0]+0.5, t[i1-1]-0.5)
    if avg is None: continue
    Zb = float(np.median(b1[i0:i1])); dZ = (avg.max()-avg.min())/1000.0
    lab = "оба канала" if state[i0] == 2 else "канал 2 ОТКЛ"
    ROWS.append(dict(state=state[i0], Zb=Zb, dZ=dZ, n=n))
    print("%-14s | %-14s | %6d | %9.2f Ом | %8.1f мОм | %9.3f %% | %+6.0f мс" %
          ("%.1f–%.1f" % (t[i0], t[i1-1]), lab, n, Zb, dZ*1000, 100*dZ/Zb, tr*1000))

both = [x for x in ROWS if x["state"] == 2]; alone = [x for x in ROWS if x["state"] == 1]
if both and alone:
    Zb_b = np.mean([x["Zb"] for x in both]); Zb_a = alone[0]["Zb"]
    dZ_b = np.mean([x["dZ"] for x in both]); dZ_a = alone[0]["dZ"]
    print()
    print("Отключение соседнего канала: BASE_1 ×%.2f (%.1f -> %.1f Ом), ΔZ_сист ×%.2f (%.0f -> %.0f мОм)"
          % (Zb_a/Zb_b, Zb_b, Zb_a, dZ_a/dZ_b, dZ_b*1000, dZ_a*1000))
    print("Относительный размах ΔZ/Z: %.3f %% при обоих каналах -> %.3f %% при одиночной паре (×%.2f)"
          % (100*dZ_b/Zb_b, 100*dZ_a/Zb_a, (dZ_a/Zb_a)/(dZ_b/Zb_b)))
    Zsh = 1.0/(1.0/Zb_b - 1.0/Zb_a)
    print()
    print("Проверка гипотезы простого параллельного шунта:")
    print("  эквивалентное сопротивление шунта %.1f Ом; при таком шунте размах ослаблялся бы" % Zsh)
    print("  в (Z_изм/Z_ист)² = %.3f раза, то есть %.0f мОм наблюдались бы как %.0f мОм."
          % ((Zb_b/Zb_a)**2, dZ_a*1000, dZ_a*(Zb_b/Zb_a)**2*1000))
    print("  Наблюдается %.0f мОм — в %.1f раза больше. Гипотеза простого шунта не описывает данные."
          % (dZ_b*1000, dZ_b/(dZ_a*(Zb_b/Zb_a)**2)))


**Анализ результатов.** Отключение соседнего канала при неизменных электродах повышает базовый импеданс трансторакального отведения в 1.71 раза (с 54.2 до 92.7 Ом) и одновременно **понижает** систолический размах в 1.85 раза (с 257 до 139 мОм). Относительный размах падает втрое: с 0.474 % до 0.150 %.

Полученное в интервальном сопоставлении значение 0.150 % согласуется с независимо измеренным в отдельной записи `14-07-01` значением 0.169 % (§1), где та же конфигурация одиночной токовой пары наблюдалась на 71 ударе против семи здесь. Согласие двух оценок, различающихся составом данных на порядок, подтверждает, что различие между конфигурациями не является случайным.

Согласованного изменения базового импеданса и размаха не наблюдается, поэтому гипотеза простого параллельного шунта данные не описывает. При эквивалентном шунте 130.4 Ом, объясняющем изменение базового импеданса, размах ослаблялся бы пропорционально квадрату отношения импедансов, то есть 139 мОм наблюдались бы как 47 мОм, тогда как измеряется 257 мОм — в 5.4 раза больше.

Отсюда следует, что подключённый соседний канал не просто добавляет параллельный путь, а **перераспределяет ток и повышает чувствительность трансторакального отведения к сердцу**. Это физически объяснимо: электроды соседнего канала расположены на боковой поверхности грудной клетки, и их подключение к входным цепям прибора направляет часть тока через область сердца.

**Результаты и умозаключения.** Межканальное влияние затрагивает не только базовый уровень, но и полезный сигнал, причём в разной степени и в разные стороны. Следовательно, измерения при обоих включённых каналах **не соответствуют модели с одной токовой парой** ни по базовому импедансу, ни по чувствительности, и сопоставлять расчёт следует с конфигурацией одиночной пары: базовый импеданс 92.7–101.4 Ом, относительный размах 0.150–0.169 %.


## §3. Верхняя граница лёгочного конфаундера

**Входные данные.** Относительное систолическое приращение удельного сопротивления лёгкого и мягких тканей из лёгочной ветки МГТУ: $\Delta\rho_2/\rho_2=0.69\,\%$ на задержке вдоха и $|\Delta\rho_1|/\rho_1\le0.04\,\%$ ([11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb) §3.4). Относительный систолический размах трансторакального отведения из §2.

**Допущения.** Прямая задача электропроводности однородна первой степени по совокупности удельных сопротивлений всех тканей: одновременное умножение их всех на общий множитель умножает импеданс на тот же множитель. По теореме Эйлера об однородных функциях отсюда следует тождество (19.1):

$$\sum_i S_i=1,\qquad S_i=\frac{\rho_i}{Z}\frac{\partial Z}{\partial\rho_i} \tag{19.1}$$

где $S_i$ — относительная чувствительность импеданса к удельному сопротивлению ткани $i$, безразмерная; суммирование ведётся по всем тканям. Каждое слагаемое неотрицательно, поскольку повышение удельного сопротивления любой области не может понизить полное сопротивление. Следовательно $0\le S_i\le 1$ для каждой ткани, и вклад ткани в относительное изменение импеданса ограничен сверху (19.2):

$$\left|\frac{\Delta Z_i}{Z}\right|=S_i\left|\frac{\Delta\rho_i}{\rho_i}\right|\le\left|\frac{\Delta\rho_i}{\rho_i}\right| \tag{19.2}$$

Эта граница не требует расчёта методом конечных элементов и не зависит от геометрии: она следует только из однородности. Ценой общности является грубость — граница достигается лишь при $S_i=1$, то есть если бы весь ток шёл исключительно через данную ткань.


In [ ]:
# @title §3 Границы вклада лёгкого и мягких тканей в трансторакальный сигнал
DRHO2_REL = 0.694/100      # Δρ2/ρ2 на задержке вдоха, 11 §3.4 (эксперимент 2, субъектная запись)
DRHO1_REL = 0.04/100       # верхняя граница |Δρ1|/ρ1, 11 §3.3

# конфигурации берутся из §1 и §2, а не вписываются вручную
conf = [("одиночная пара, запись 14-07-01 (71 уд.)", 100*REC["14-07-01"]["dZ"]/REC["14-07-01"]["Zb"]),
        ("одиночная пара, интервал 14-17-16 (7 уд.)", 100*alone[0]["dZ"]/alone[0]["Zb"]),
        ("оба канала, запись 14-19-52 (69 уд.)",      100*REC["14-19-52"]["dZ"]/REC["14-19-52"]["Zb"])]

print("Верхние границы по (19.2), не требующие расчёта МКЭ:")
print("  лёгкое:        |ΔZ/Z| ≤ %.3f %%" % (100*DRHO2_REL))
print("  мягкие ткани:  |ΔZ/Z| ≤ %.3f %%" % (100*DRHO1_REL))
print()
print("%-38s | %11s | %14s | %s" % ("конфигурация измерения", "ΔZ/Z изм.", "граница/сигнал", "вывод"))
print("-"*96)
for name, meas in conf:
    ratio = 100*DRHO2_REL/meas
    verdict = ("конфаундер может превышать сигнал" if ratio > 1
               else "конфаундер заведомо меньше сигнала")
    print("%-38s | %10.3f %% | %13.1f× | %s" % (name, meas, ratio, verdict))
print()
print("Требуемая точность S_лёгкого: чтобы остаток был определён с точностью 20 %,")
for name, meas in conf:
    print("  при ΔZ/Z = %.3f %% нужно знать S_лёгкого с абсолютной точностью %.3f"
          % (meas, 0.20*meas/100/DRHO2_REL))


**Анализ результатов.** Верхняя граница лёгочного вклада составляет 0.694 % относительного изменения импеданса. Измеренный относительный размах в конфигурации, соответствующей модели, равен 0.169 % по записи `14-07-01` и 0.150 % по интервалу записи `14-17-16`, то есть **граница превышает весь измеренный сигнал в четыре с лишним раза**. В конфигурации с обоими включёнными каналами, где размах вдвое больше, граница по-прежнему вдвое превышает сигнал.

Вклад мягких тканей ограничен величиной 0.04 %, то есть не более четверти сигнала в модельной конфигурации. Поскольку сама эта величина является границей, а не оценкой ([11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb) §3.3), вкладом мягких тканей допустимо пренебречь, оговорив его как остаточную неопределённость.

Требуемая точность знания чувствительности к лёгкому получается обращением (19.2): чтобы остаток после вычитания был определён с точностью 20 %, чувствительность $S_{\text{лёгкого}}$ должна быть известна с абсолютной точностью около 0.05 в модельной конфигурации. Требование жёсткое: величина заключена между нулём и единицей, то есть речь идёт о знании с погрешностью в несколько процентных пунктов.

**Результаты и умозаключения.** Пренебрежение лёгочным конфаундером в трансторакальном отведении **недопустимо**: его верхняя граница превышает измеряемый сердечный сигнал. Вместе с тем сама граница слишком груба, чтобы служить поправкой, поскольку получена в предположении, что весь ток идёт через лёгкое.

Отсюда задание на расчёт методом конечных элементов, единственную величину которого нельзя получить аналитически: требуется относительная чувствительность трансторакального отведения к удельному сопротивлению лёгкого на геометрии, полученной по томографии испытуемого, с погрешностью не хуже 0.05. Постановка приведена в §4.


## §4. Задание на расчёт методом конечных элементов

Расчёт выполняется в `MATLAB_TRKG4_real_subjects` на сетке, построенной по индивидуальной томографии добровольца; пункт 3 плана работ той папки («проверка расчётных rho для тканей») описывает ровно нужную операцию.

**Что задаётся.** Геометрия тела, лёгких, сердца и костей из сегментации `3D_Slicer`; положение четырёх электродов трансторакального отведения; удельные сопротивления тканей, для мягких тканей и лёгкого — из [09](33.01_Статическая_оценка_параметров_боковых_сборок.ipynb), для крови и кости — литературные ([13](00.01_Методологический_контракт.md) §3).

**Что вычисляется.**

1. Базовый импеданс отведения при одиночной токовой паре. **Проверка модели:** он должен составить 90–105 Ом, что согласуется с двумя независимыми измерениями РНЦХ при отключённом соседнем канале: 92.7 Ом в интервале записи `14-17-16` и 101.4 Ом в записи `14-07-01` (§1, §2). Совпадение подтверждает геометрию и назначение тканей и даёт право доверять вычисленным далее чувствительностям.
2. Относительные чувствительности $S_i$ к удельному сопротивлению каждой ткани — возмущением по одной ткани за раз. **Контроль:** сумма по всем тканям должна равняться единице с точностью счёта, что является прямой проверкой (19.1).
3. Чувствительность импеданса к изменению объёма сердца при фиксированных удельных сопротивлениях — производная, необходимая для обратной задачи по геометрии.

**Чего расчёт не заменяет.** Межканальное влияние (§2) описывается только двумя токовыми парами и в постановке с одной парой не воспроизводится. Поэтому сопоставлять расчёт следует с интервалом одиночной пары, а не с записями при обоих включённых каналах.

**Порядок дальнейших действий.** После получения $S_{\text{лёгкого}}$ вычитание конфаундера выполняется по (19.3):

$$\left(\frac{\Delta Z}{Z}\right)_{\text{сердце}}=\left(\frac{\Delta Z}{Z}\right)_{\text{изм}}-S_{\text{лёгкого}}\frac{\Delta\rho_2}{\rho_2}-S_{\text{мягк}}\frac{\Delta\rho_1}{\rho_1} \tag{19.3}$$

после чего остаток обращается в изменение геометрии сердца. Обратную задачу следует ставить не на сфере, а на эллипсоиде с формой и ориентацией, закреплёнными по томографии, и единственным неизвестным масштабом: обоснование — в [18](70.09_Импедансный_критерий_эквивалентной_сферы.ipynb) §4, где показано, что при контрасте проводимостей 15.9 и вытянутости 2:1 замена на равнообъёмную сферу искажает дипольный отклик на 66 %.


## §5. Выводы

1. **Трансторакальный сердечно-синхронный сигнал выделен** по трём записям РНЦХ. Провал усреднённой волны наступает через 250–355 мс после зубца R, что согласуется по порядку с задержкой в лёгочной ветке и подтверждает сердечное происхождение составляющей.

2. **Единый порог детекции зубцов R непригоден**: качество электрокардиограммы между записями различается, рабочие пороги составляют 0.8 и 1.5. Контроль качества должен включать не только согласие двух оценок частоты сердечных сокращений, но и её физиологическую правдоподобность: проверка одной лишь согласованности пропускает случай, когда детектор устойчиво находит малую долю зубцов, поскольку тогда занижены обе оценки.

3. **Межканальное влияние затрагивает и полезный сигнал.** Отключение соседнего канала при неизменных электродах повышает базовый импеданс в 1.71 раза и понижает систолический размах в 1.85 раза; относительный размах падает втрое. Гипотеза простого параллельного шунта данные не описывает — расхождение в 5.4 раза. Подключённый соседний канал перераспределяет ток и повышает чувствительность отведения к сердцу.

4. **Модели соответствует конфигурация с одиночной токовой парой:** базовый импеданс 92.7–101.4 Ом, относительный размах 0.150–0.169 %. Две независимые оценки — по отдельной записи на 71 ударе и по интервалу на семи — согласуются между собой.

5. **Пренебрежение лёгочным конфаундером недопустимо.** Его верхняя граница, полученная из однородности прямой задачи без расчёта МКЭ, составляет 0.694 % и превышает весь измеренный сердечный сигнал более чем вчетверо. Вкладом мягких тканей (граница 0.04 %) пренебречь допустимо с оговоркой.

6. **Ключевая недостающая величина — относительная чувствительность к лёгкому**, и требуется она с абсолютной точностью около 0.05. Это единственное, что нельзя получить аналитически, и это определяет содержание расчёта МКЭ (§4).

7. **Главное ограничение имеющихся данных** — записи РНЦХ сняты не по дыхательным состояниям: приращения Δρ измерены на задержке вдоха, тогда как трансторакальные ансамбли построены по всей записи. Согласование состояний, обязательность которого показана в [11](33.03_Пульсовые_ансамбли_боковых_сборок.ipynb) §3.4, здесь не выполнено, и это следует устранить при следующей съёмке: запись с одиночной токовой парой на задержке вдоха занимает менее минуты.
